In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os
from kaggle_secrets import UserSecretsClient
from google.genai import types

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    print("API key loaded successfully.")
except:
    print("Error: API key missing.")

API key loaded successfully.


In [2]:
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters
from google.adk.tools.function_tool import FunctionTool

print("ADK import successful.")

ADK import successful.


In [3]:
mcp_image_server = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command="npx",
            args=[
                "-y",
                "@modelcontextprotocol/server-everything"
            ],
            tool_filter=["getTinyImage"],
        ),
        timeout=30,
    )
)

print("MCP tool ready.")

MCP tool ready.


In [4]:
def check_order_status(order_id: str):
    return {
        "order_id": order_id,
        "status": "Delivered",
        "message": f"Order {order_id} is delivered successfully."
    }

order_status_tool = FunctionTool(func=check_order_status)

print("Custom tool ready.")

Custom tool ready.


In [5]:
# ORDER STATUS TOOL
def check_order_status(order_id: str):
    return {
        "order_id": order_id,
        "status": "Delivered",
        "message": f"Order {order_id} is delivered successfully."
    }

order_status_tool = FunctionTool(func=check_order_status)



# PRODUCT AVAILABILITY TOOL
def check_product_availability(product_name: str):
    fake_db = {
        "samsung s24": {"stock": 12, "status": "In Stock"},
        "realme 12 pro": {"stock": 0, "status": "Out of Stock"},
        "vivo v29": {"stock": 4, "status": "Low Stock"},
        "oppo f25": {"stock": 2, "status": "Low Stock"},
        "iphone 15": {"stock": 0, "status": "Out of Stock"},
        "redmi note 13": {"stock": 18, "status": "In Stock"}
    }

    key = product_name.lower()

    if key not in fake_db:
        return {
            "product": product_name,
            "availability": "Not Found",
            "message": f"{product_name} is not in our catalog."
        }

    info = fake_db[key]

    return {
        "product": product_name,
        "availability": info["status"],
        "stock": info["stock"],
        "message": f"{product_name} is {info['status']} (Stock: {info['stock']})."
    }

product_tool = FunctionTool(func=check_product_availability)

print("Order tool + Product tool ready.")

Order tool + Product tool ready.


In [6]:
agent = LlmAgent(
    model=Gemini(model="gemini-2.5-flash-lite"),
    name="cs_agent",
    instruction="""
    You are a customer support agent for an online store.

    Rules:
    - If user provides an order_id → use order status tool.
    - If user asks about product availability → use product availability tool.
    - If user asks for product image → use MCP tiny image tool.
    - Keep replies short and support-style.
    - Handle multiple tasks step-by-step (sequential).
    - Remember user info through session memory.
    """,
    tools=[mcp_image_server, order_status_tool, product_tool]
)

print("Customer Support Agent created.")

Customer Support Agent created.


In [7]:
session_service = InMemorySessionService()
runner = Runner(agent=agent, session_service=session_service, app_name="cs_app")

print("Runner ready.")

Runner ready.


In [12]:
await runner.run_debug("My name is Karishma Jain and I want to buy iPhone 17.", verbose=True)
await runner.run_debug("What's my name and what I want to buy?", verbose=True)


 ### Continue session: debug_session_id

User > My name is Karishma Jain and I want to buy iPhone 17.


cs_agent > [Calling tool: check_product_availability({'product_name': 'iPhone 17'})]
cs_agent > [Tool result: {'product': 'iPhone 17', 'availability': 'Not Found', 'message': 'iPhone 17 is not in our catalog.'}]
cs_agent > I am sorry, Karishma Jain, the iPhone 17 is not in our catalog.

 ### Continue session: debug_session_id

User > What's my name and what I want to buy?
cs_agent > Your name is Karishma Jain and you want to buy an iPhone 17.


[Event(model_version='gemini-2.5-flash-lite', content=Content(
   parts=[
     Part(
       text='Your name is Karishma Jain and you want to buy an iPhone 17.'
     ),
   ],
   role='model'
 ), grounding_metadata=None, partial=None, turn_complete=None, finish_reason=<FinishReason.STOP: 'STOP'>, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=GenerateContentResponseUsageMetadata(
   candidates_token_count=17,
   prompt_token_count=1245,
   prompt_tokens_details=[
     ModalityTokenCount(
       modality=<MediaModality.TEXT: 'TEXT'>,
       token_count=1245
     ),
   ],
   total_token_count=1262
 ), live_session_resumption_update=None, input_transcription=None, output_transcription=None, avg_logprobs=None, logprobs_result=None, cache_metadata=None, citation_metadata=None, invocation_id='e-9e35629d-13f5-4248-88a1-dc9d99012809', author='cs_agent', actions=EventActions(skip_summarization=None, state_delta={}, artifact_delta={}, transfer_to_agent=

In [9]:
await runner.run_debug("Check my order 2024", verbose=True)


 ### Continue session: debug_session_id

User > Check my order 2024


cs_agent > [Calling tool: check_order_status({'order_id': '2024'})]
cs_agent > [Tool result: {'order_id': '2024', 'status': 'Delivered', 'message': 'Order 2024 is delivered successfully.'}]
cs_agent > Your order 2024 has been delivered.


[Event(model_version='gemini-2.5-flash-lite', content=Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'order_id': '2024'
         },
         id='adk-60e1eaf2-51c9-4e2a-8986-819767e7dc2e',
         name='check_order_status'
       )
     ),
   ],
   role='model'
 ), grounding_metadata=None, partial=None, turn_complete=None, finish_reason=<FinishReason.STOP: 'STOP'>, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=GenerateContentResponseUsageMetadata(
   candidates_token_count=22,
   prompt_token_count=844,
   prompt_tokens_details=[
     ModalityTokenCount(
       modality=<MediaModality.TEXT: 'TEXT'>,
       token_count=844
     ),
   ],
   total_token_count=866
 ), live_session_resumption_update=None, input_transcription=None, output_transcription=None, avg_logprobs=None, logprobs_result=None, cache_metadata=None, citation_metadata=None, invocation_id='e-91450d2f-e64a-4980-8485-6baeea7d3410', aut

In [10]:
await runner.run_debug("is samsung s24 available?", verbose=True)


 ### Continue session: debug_session_id

User > is samsung s24 available?


cs_agent > [Calling tool: check_product_availability({'product_name': 'samsung s24'})]
cs_agent > [Tool result: {'product': 'samsung s24', 'availability': 'In Stock', 'stock': 12, 'message': 'samsung s24 is In St...]
cs_agent > Yes, the samsung s24 is in stock.


[Event(model_version='gemini-2.5-flash-lite', content=Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'product_name': 'samsung s24'
         },
         id='adk-3249936d-a719-45e2-99d9-c70866be7f33',
         name='check_product_availability'
       )
     ),
   ],
   role='model'
 ), grounding_metadata=None, partial=None, turn_complete=None, finish_reason=<FinishReason.STOP: 'STOP'>, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=GenerateContentResponseUsageMetadata(
   candidates_token_count=23,
   prompt_token_count=929,
   prompt_tokens_details=[
     ModalityTokenCount(
       modality=<MediaModality.TEXT: 'TEXT'>,
       token_count=929
     ),
   ],
   total_token_count=952
 ), live_session_resumption_update=None, input_transcription=None, output_transcription=None, avg_logprobs=None, logprobs_result=None, cache_metadata=None, citation_metadata=None, invocation_id='e-b03b2ff4-671b-41cc-8f72

In [14]:
await runner.run_debug("Check order 500 and show product image", verbose=True)


 ### Continue session: debug_session_id

User > Check order 500 and show product image


cs_agent > [Calling tool: check_order_status({'order_id': '500'})]
cs_agent > [Tool result: {'order_id': '500', 'status': 'Delivered', 'message': 'Order 500 is delivered successfully.'}]


cs_agent > [Calling tool: getTinyImage({})]
cs_agent > [Tool result: {'content': [{'type': 'text', 'text': 'This is a tiny image:'}, {'type': 'image', 'data': 'iVBORw0KG...]
cs_agent > Your order 500 has been delivered. Here is a tiny image.


[Event(model_version='gemini-2.5-flash-lite', content=Content(
   parts=[
     Part(
       function_call=FunctionCall(
         args={
           'order_id': '500'
         },
         id='adk-0df4385c-000d-4b94-a4c7-e9614ba117df',
         name='check_order_status'
       )
     ),
   ],
   role='model'
 ), grounding_metadata=None, partial=None, turn_complete=None, finish_reason=<FinishReason.STOP: 'STOP'>, error_code=None, error_message=None, interrupted=None, custom_metadata=None, usage_metadata=GenerateContentResponseUsageMetadata(
   candidates_token_count=21,
   prompt_token_count=1355,
   prompt_tokens_details=[
     ModalityTokenCount(
       modality=<MediaModality.TEXT: 'TEXT'>,
       token_count=1355
     ),
   ],
   total_token_count=1376
 ), live_session_resumption_update=None, input_transcription=None, output_transcription=None, avg_logprobs=None, logprobs_result=None, cache_metadata=None, citation_metadata=None, invocation_id='e-2c8867d6-0fe9-4554-8c12-f61321b426a6', a